# Launch-Deets: setup and walkthrough

Select the project `.venv` kernel and run from top to bottom. Local inspection runs by default. Set `RUN_LIVE=True` for connection checks and a sample request; these use provider credits. Set `RUN_INGEST=True` only when uploading the corpus. Evaluation and token inspection are in notebook 02.


In [ ]:
from pathlib import Path
import sys
APP_ROOT = Path.cwd()
if not (APP_ROOT / 'data').exists():
    APP_ROOT = Path.cwd().parent
sys.path.insert(0, str(APP_ROOT / 'src'))
from launch_deets.data import load_archive, validate_archive, split_documents
from launch_deets.config import Settings
archive = load_archive(APP_ROOT)
print({'records': len(archive.records), 'documents': len(archive.documents), 'relationships': len(archive.relationships)})
import os
from launch_deets.config import load_local_env
load_local_env(override=True)
RUN_LIVE = False
import os
if not RUN_LIVE:
    os.environ["LANGSMITH_TRACING"] = "false"
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
RUN_INGEST = False
settings = Settings.from_env()


## 0. Neo4j connection check

Select the project `.venv` kernel and run the setup cell above first.
This follows the course GraphDatabase.driver pattern, loads credentials from `.env`, and checks connectivity.


In [ ]:
if RUN_LIVE:
    import os
    from launch_deets.config import load_local_env
    from neo4j import GraphDatabase
    
    load_local_env(override=True)
    
    driver = GraphDatabase.driver(
        os.environ["NEO4J_URI"],
        auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
    )
    driver.verify_connectivity()
    print("Neo4j connected successfully!")
    driver.close()
else:
    print("Live connection check disabled.")


### Chat and embedding model checks

This sends one short prompt to the configured chat model and embeds one short sentence. It prints the reply and embedding dimensions, not the full vector.


In [ ]:
if RUN_LIVE:
    from openai import OpenAI
    with OpenAI(
        api_key=os.environ["NEBIUS_API_KEY"],
        base_url=os.environ["NEBIUS_BASE_URL"],
    ) as nebius:
        chat_result = nebius.chat.completions.create(
            model=os.environ["NEBIUS_MODEL"],
            messages=[{"role": "user", "content": "Reply with exactly OK."}],
            max_tokens=4,
        )
        embedding_result = nebius.embeddings.create(
            model=os.environ["NEBIUS_EMBEDDING_MODEL"],
            input="A short test sentence for embeddings.",
        )
    
    print(f"Chat model: {chat_result.model} â€” {chat_result.choices[0].message.content}")
    print(f"Embedding model: {embedding_result.model} â€” {len(embedding_result.data[0].embedding)} dimensions")
else:
    print("Live connection check disabled.")


### 0b. LangSmith connection check

Run the setup cell first. This read-only request checks the API key.

In [ ]:
if RUN_LIVE:
    import os
    from launch_deets.config import load_local_env
    from urllib.request import Request, urlopen
    
    load_local_env(override=True)
    
    request = Request(
        "https://api.smith.langchain.com/api/v1/sessions?limit=1",
        headers={"x-api-key": os.environ["LANGSMITH_API_KEY"]},
    )
    with urlopen(request, timeout=20) as response:
        response.read()
    
    print("LangSmith connected successfully!")
else:
    print("Live connection check disabled.")


### 0c. TypeSafe connection check

Run the setup cell first. This sends one small `system_one` request using `TYPESAFE_API_KEY` from `.env`; it may use a small amount of TypeSafe credit.


In [ ]:
if RUN_LIVE:
    import os
    from launch_deets.config import load_local_env
    from typesafe_sdk import Noul, TypeSafeClient
    
    load_local_env(override=True)
    
    with TypeSafeClient(api_key=os.environ["TYPESAFE_API_KEY"]) as client:
        response = client.system_one(
            state={"message": "TypeSafe connection check"},
            questions={"connected": Noul(instructions="Does the message say this is a TypeSafe connection check?")},
        )
    
    print("TypeSafe connected successfully!")
    print(f"NOUL response: {response.nouls['connected'].noul:.2f}")
else:
    print("Live connection check disabled.")


## 1. Validate IDs, metadata, and provenance
No external service calls happen in this checkpoint.

In [ ]:
errors = validate_archive(archive)
assert not errors, errors
archive.records.head(8)

## 2. Inspect chunks and inherited metadata
Baseline starts at 500 characters with 100 character overlap; inspect boundaries before changing it.

In [ ]:
chunks = split_documents(archive.documents, chunk_size=500, overlap=100)
[(c['metadata']['document_id'], c['metadata']['launch_id'], c['metadata']['start_index'], c['text'][:180]) for c in chunks[:8]]

## 3. Structured, sparse, and graph baselines
CSV and BM25 use local synthetic data. Graph fallback reads relationships.csv; Neo4j is optional.

In [ ]:
from launch_deets.retrieval import HybridIndex, structured_lookup, graph_paths
index = HybridIndex(archive)
structured_lookup(archive, 'Alpha telemetry packet loss 41 seconds')[:3]

In [ ]:
index.bm25('What did firmware C8 change for the Charlie inertial measurement unit?', k=5)

In [ ]:
graph_paths(archive, 'LAUNCH-L7-CHARLIE')

## Upload graph relationships (optional)
Uses the same Neo4j MERGE operation as the application. Enable both live calls and ingestion explicitly.


In [ ]:
from launch_deets.providers import neo4j_ingest
if RUN_LIVE and RUN_INGEST:
    display(neo4j_ingest(settings, archive))
else:
    print("Graph upload disabled.")


## Inspect a graph path
A launch connects to a component, an incident, and a corrective action. Each edge retains evidence IDs.


In [ ]:
from launch_deets.providers import neo4j_paths
if RUN_LIVE:
    display(neo4j_paths(settings, "LAUNCH-L7-CHARLIE"))
else:
    print("Hosted graph query disabled; see local path above.")


## Upload document vectors (optional)
Run only after creating the Pinecone index with the configured embedding dimension. Re-upload when the corpus or chunk settings change.


In [ ]:
settings = Settings.from_env()
{'nebius_model_configured': bool(settings.nebius_model), 'embedding_model_configured': bool(settings.nebius_embedding_model), 'pinecone_configured': bool(settings.pinecone_api_key and settings.pinecone_index_name), 'neo4j_configured': bool(settings.neo4j_uri and settings.neo4j_password), 'typesafe_configured': bool(settings.typesafe_api_key), 'langsmith_configured': bool(settings.langsmith_api_key)}

In [ ]:
from launch_deets.providers import pinecone_ingest
if RUN_LIVE and RUN_INGEST:
    display(pinecone_ingest(settings, archive))
else:
    print("Vector upload disabled.")


## Ask the agent
NOUL checks the question first. The agent selects relevant retrieval tools and generates an answer from their evidence.


In [ ]:
from launch_deets.agent import ask
if RUN_LIVE:
    display(ask("Which component was tied to the Charlie yaw incident, and what fixed it?", archive=archive, settings=settings))
else:
    print("Live request disabled.")
